In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")

train_df = pd.read_pickle(DATA_DIR / "train.pkl")
valid_df = pd.read_pickle(DATA_DIR / "valid.pkl")
test_df  = pd.read_pickle(DATA_DIR / "test.pkl")

target_col = "Money Laundering Risk Score"

y_train = train_df[target_col].astype(int)
y_valid = valid_df[target_col].astype(int)
y_test  = test_df[target_col].astype(int)

In [8]:
experiments = []

In [9]:
experiments.append(
    {
        "model_name": "baseline",
        "hypothesis": "Простая логистическая регрессия без feature engineering задаёт отправную точку для классификации уровней риска.",
        "params": "LogisticRegression (solver=lbfgs, max_iter=1000), без OHE",
        "split": "validation",
        "accuracy": 0.10266666666666667,
        "f1_weighted": 0.07419634496211962,
        "comment": "Нулевой baseline: качество чуть выше случайного, сильно ниже ожидаемого.",
    }
)

experiments.append(
    {
        "model_name": "baseline",
        "hypothesis": "Проверить обобщающую способность baseline на test.",
        "params": "LogisticRegression (solver=lbfgs, max_iter=1000), без OHE",
        "split": "test",
        "accuracy": 0.11333333333333333,
        "f1_weighted": 0.08175977975488229,
        "comment": "Качество на test аналогично validation, baseline остаётся слабым.",
    }
)

experiments_df = pd.DataFrame(experiments)
experiments_df

,model_name,hypothesis,params,split,accuracy,f1_weighted,comment
0,baseline,Простая логистическая регрессия без feature en...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.102667,0.074196,Нулевой baseline: качество чуть выше случайног...
1,baseline,Проверить обобщающую способность baseline на t...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.113333,0.081760,"Качество на test аналогично validation, baseli..."


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [11]:
categorical_features = [
    "Country",
    "Transaction Type",
    "Industry",
    "Destination Country",
    "Tax Haven Country",
    "Financial Institution_grouped",
]

numeric_features = [
    "Shell Companies Involved",
    "Amount (USD)",
    "transaction_year",
    "transaction_month",
    "transaction_dayofweek",
    "transaction_hour",
    "is_illegal",
    "is_reported",
]

X_train_raw = train_df[numeric_features + categorical_features].copy()
X_valid_raw = valid_df[numeric_features + categorical_features].copy()
X_test_raw  = test_df[numeric_features + categorical_features].copy()

y_train = train_df[target_col].astype(int)
y_valid = valid_df[target_col].astype(int)
y_test  = test_df[target_col].astype(int)

In [12]:
X_train_ohe = pd.get_dummies(X_train_raw, columns=categorical_features, drop_first=False)
X_valid_ohe = pd.get_dummies(X_valid_raw, columns=categorical_features, drop_first=False)
X_test_ohe  = pd.get_dummies(X_test_raw,  columns=categorical_features, drop_first=False)

X_valid_ohe = X_valid_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)
X_test_ohe  = X_test_ohe.reindex(columns=X_train_ohe.columns, fill_value=0)

X_train_ohe.shape, X_valid_ohe.shape, X_test_ohe.shape

((7000, 67), (1500, 67), (1500, 67))

In [13]:
RANDOM_STATE = 42
logreg_ohe = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE
)

logreg_ohe.fit(X_train_ohe, y_train)

valid_pred_logreg = logreg_ohe.predict(X_valid_ohe)
test_pred_logreg  = logreg_ohe.predict(X_test_ohe)

valid_acc_logreg = accuracy_score(y_valid, valid_pred_logreg)
test_acc_logreg  = accuracy_score(y_test,  test_pred_logreg)

valid_f1_logreg = f1_score(y_valid, valid_pred_logreg, average="weighted")
test_f1_logreg  = f1_score(y_test,  test_pred_logreg,  average="weighted")

valid_acc_logreg, valid_f1_logreg, test_acc_logreg, test_f1_logreg

(0.10733333333333334,
 0.07038163081405935,
 0.10866666666666666,
 0.07211124649267331)

In [28]:
experiments.append(
    {
        "model_name": "LogisticRegression_OHE",
        "hypothesis": "Добавление one-hot кодирования категориальных признаков улучшит качество по сравнению с baseline без OHE.",
        "params": "LogisticRegression (solver=lbfgs, max_iter=1000) + OHE для категориальных признаков",
        "split": "validation",
        "accuracy": valid_acc_logreg,
        "f1_weighted": valid_f1_logreg,
        "comment": "Логистическая регрессия на полном OHE-наборе признаков.",
    }
)

experiments.append(
    {
        "model_name": "LogisticRegression_OHE",
        "hypothesis": "Проверка обобщающей способности OHE-модели на test.",
        "params": "LogisticRegression (solver=lbfgs, max_iter=1000) + OHE",
        "split": "test",
        "accuracy": test_acc_logreg,
        "f1_weighted": test_f1_logreg,
        "comment": "Test-качество для модели с OHE.",
    }
)

experiments_df = pd.DataFrame(experiments)
experiments_df

,model_name,hypothesis,params,split,accuracy,f1_weighted,comment
0,baseline,Простая логистическая регрессия без feature en...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.102667,0.074196,Нулевой baseline: качество чуть выше случайног...
1,baseline,Проверить обобщающую способность baseline на t...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.113333,0.081760,"Качество на test аналогично validation, baseli..."
2,LogisticRegression_OHE,Добавление one-hot кодирования категориальных ...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.107333,0.070382,Логистическая регрессия на полном OHE-наборе п...
3,LogisticRegression_OHE,Проверка обобщающей способности OHE-модели на ...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.108667,0.072111,Test-качество для модели с OHE.
4,KNN_OHE,Подбор количества соседей k улучшит качество K...,Pipeline(StandardScaler(with_mean=False) + KNe...,validation,0.112000,0.111803,Лучший k по weighted F1 на validation: k=3.
5,KNN_OHE,Проверка обобщения лучшей настройки k на test.,"k=3, weights='distance'",test,0.113333,0.113073,Test-результат для лучшего k.
6,LogisticRegression_OHE,Добавление one-hot кодирования категориальных ...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.107333,0.070382,Логистическая регрессия на полном OHE-наборе п...
7,LogisticRegression_OHE,Проверка обобщающей способности OHE-модели на ...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.108667,0.072111,Test-качество для модели с OHE.


In [24]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np

k_values = list(range(1, 10))
weight_values = ["uniform", "distance"]
p_values = [1, 2]

big_results = []

for k in k_values:
    for w in weight_values:
        for p in p_values:
            clf = Pipeline(
                steps=[
                    ("scaler", StandardScaler(with_mean=False)),
                    ("model", KNeighborsClassifier(
                        n_neighbors=k,
                        weights=w,
                        metric="minkowski",
                        p=p,
                    )),
                ]
            )

            clf.fit(X_train_ohe, y_train)

            valid_pred = clf.predict(X_valid_ohe)
            test_pred  = clf.predict(X_test_ohe)

            valid_acc = accuracy_score(y_valid, valid_pred)
            test_acc  = accuracy_score(y_test,  test_pred)

            valid_f1 = f1_score(y_valid, valid_pred, average="weighted")
            test_f1  = f1_score(y_test,  test_pred,  average="weighted")

            big_results.append(
                {
                    "k": k,
                    "weights": w,
                    "p": p,
                    "valid_acc": valid_acc,
                    "valid_f1": valid_f1,
                    "test_acc": test_acc,
                    "test_f1": test_f1,
                }
            )

big_results_df = pd.DataFrame(big_results)
big_results_df.sort_values("valid_f1", ascending=False).head(10)

,k,weights,p,valid_acc,valid_f1,test_acc,test_f1
15,4,distance,2,0.116000,0.115699,0.106000,0.105685
1,1,uniform,2,0.112667,0.112250,0.112667,0.112174
7,2,distance,2,0.112667,0.112250,0.112667,0.112174
3,1,distance,2,0.112667,0.112250,0.112667,0.112174
11,3,distance,2,0.112000,0.111803,0.113333,0.113073
26,7,distance,1,0.110000,0.109735,0.105333,0.104596
21,6,uniform,2,0.114000,0.108348,0.096000,0.094471
0,1,uniform,1,0.106000,0.105427,0.108000,0.107649
2,1,distance,1,0.106000,0.105427,0.108000,0.107649
6,2,distance,1,0.106000,0.105427,0.108000,0.107649


In [ ]:
experiments.append(
    {
        "model_name": "KNN_OHE_big_grid",
        "hypothesis": "Расширенная настройка гиперпараметров KNN (k, weights, p) улучшит качество по сравнению с базовым KNN и линейной моделью.",
        "params": "Pipeline(StandardScaler(with_mean=False) + "
                  "KNeighborsClassifier(n_neighbors=15, weights='distance', metric='minkowski', p=2)); "
                  "k ∈ {1,3,…,39}, weights ∈ {uniform,distance}, p ∈ {1,2}",
        "split": "validation",
        "accuracy": 0.116000,
        "f1_weighted": 0.115699,
        "comment": "Лучший набор гиперпараметров по weighted F1 на validation; заметное, но небольшое улучшение относительно baseline.",
    }
)
experiments.append(
    {
        "model_name": "KNN_OHE_big_grid",
        "hypothesis": "Проверка обобщающей способности лучшей конфигурации KNN на тестовом наборе.",
        "params": "n_neighbors=15, weights='distance', metric='minkowski', p=2",
        "split": "test",
        "accuracy": 0.106000,
        "f1_weighted": 0.105685,
        "comment": "Качество на test ниже, чем на validation, что указывает на возможное переобучение к валидации.",
    }
)

In [39]:
experiments_df = pd.DataFrame(experiments)
experiments_df

,model_name,hypothesis,params,split,accuracy,f1_weighted,comment
0,baseline,Простая логистическая регрессия без feature en...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.102667,0.074196,Нулевой baseline: качество чуть выше случайног...
1,baseline,Проверить обобщающую способность baseline на t...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.113333,0.081760,"Качество на test аналогично validation, baseli..."
2,LogisticRegression_OHE,Добавление one-hot кодирования категориальных ...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.107333,0.070382,Логистическая регрессия на полном OHE-наборе п...
3,LogisticRegression_OHE,Проверка обобщающей способности OHE-модели на ...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.108667,0.072111,Test-качество для модели с OHE.
4,KNN_OHE_big_grid,"Расширенная настройка гиперпараметров KNN (k, ...",Pipeline(StandardScaler(with_mean=False) + KNe...,validation,0.116000,0.115699,Лучший набор гиперпараметров по weighted F1 на...
5,KNN_OHE_big_grid,Проверка обобщающей способности лучшей конфигу...,"n_neighbors=15, weights='distance', metric='mi...",test,0.106000,0.105685,"Качество на test ниже, чем на validation, что ..."


In [40]:
from sklearn.ensemble import RandomForestClassifier

rf_params = []

for n_estimators in [50, 100, 200]:
    for max_depth in [None, 10, 20]:
        for min_samples_split in [2, 5]:
            for min_samples_leaf in [1, 2]:
                for max_features in ["sqrt", "log2"]:
                    rf_params.append(
                        {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "min_samples_split": min_samples_split,
                            "min_samples_leaf": min_samples_leaf,
                            "max_features": max_features,
                        }
                    )

rf_results = []

for p in rf_params:
    rf = RandomForestClassifier(
        n_estimators=p["n_estimators"],
        max_depth=p["max_depth"],
        min_samples_split=p["min_samples_split"],
        min_samples_leaf=p["min_samples_leaf"],
        max_features=p["max_features"],
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train_ohe, y_train)

    valid_pred = rf.predict(X_valid_ohe)
    test_pred  = rf.predict(X_test_ohe)

    valid_acc = accuracy_score(y_valid, valid_pred)
    test_acc  = accuracy_score(y_test,  test_pred)

    valid_f1 = f1_score(y_valid, valid_pred, average="weighted")
    test_f1  = f1_score(y_test,  test_pred,  average="weighted")

    rf_results.append(
        {
            **p,
            "valid_acc": valid_acc,
            "valid_f1": valid_f1,
            "test_acc": test_acc,
            "test_f1": test_f1,
        }
    )

rf_results_df = pd.DataFrame(rf_results)
rf_results_df.sort_values("valid_f1", ascending=False).head(10)

,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,valid_acc,valid_f1,test_acc,test_f1
48,200,NaN,2,1,sqrt,0.122000,0.121684,0.106667,0.106278
40,100,20.0,2,1,sqrt,0.117333,0.116865,0.106000,0.104398
24,100,NaN,2,1,sqrt,0.117333,0.116620,0.097333,0.097396
0,50,NaN,2,1,sqrt,0.116667,0.115287,0.091333,0.090800
27,100,NaN,2,2,log2,0.116000,0.115012,0.116000,0.114608
3,50,NaN,2,2,log2,0.114667,0.113601,0.116667,0.116076
49,200,NaN,2,1,log2,0.114000,0.112886,0.109333,0.108115
68,200,20.0,5,1,sqrt,0.115333,0.112835,0.104000,0.102929
20,50,20.0,5,1,sqrt,0.112000,0.111703,0.098667,0.098066
2,50,NaN,2,2,sqrt,0.113333,0.111285,0.115333,0.113611


In [41]:
experiments.append(
    {
        "model_name": "RandomForest_OHE_big_grid",
        "hypothesis": "Расширенная настройка гиперпараметров случайного леса (число деревьев, глубина, минимальное число объектов в узле и число признаков при разбиении) позволит улучшить качество по сравнению с базовой моделью и KNN.",
        "params": (
            "RandomForestClassifier(n_estimators=200, max_depth=None, "
            "min_samples_split=2, min_samples_leaf=1, max_features='sqrt', random_state=42); "
            "n_estimators ∈ {50,100,200}, max_depth ∈ {None,10,20}, "
            "min_samples_split ∈ {2,5}, min_samples_leaf ∈ {1,2}, max_features ∈ {'sqrt','log2'}"
        ),
        "split": "validation",
        "accuracy": 0.122000,
        "f1_weighted": 0.121684,
        "comment": "Лучший набор гиперпараметров по weighted F1 на validation; ансамбль даёт наибольшее качество среди рассмотренных моделей.",
    }
)

In [42]:
experiments.append(
    {
        "model_name": "RandomForest_OHE_big_grid",
        "hypothesis": "Проверка обобщающей способности лучшей конфигурации случайного леса на тестовом наборе.",
        "params": "n_estimators=200, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features='sqrt'",
        "split": "test",
        "accuracy": 0.106667,
        "f1_weighted": 0.106278,
        "comment": "На тесте качество ниже, чем на validation, возможное переобучение к валидации.",
    }
)

In [43]:
experiments_df = pd.DataFrame(experiments)
experiments_df

,model_name,hypothesis,params,split,accuracy,f1_weighted,comment
0,baseline,Простая логистическая регрессия без feature en...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.102667,0.074196,Нулевой baseline: качество чуть выше случайног...
1,baseline,Проверить обобщающую способность baseline на t...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.113333,0.081760,"Качество на test аналогично validation, baseli..."
2,LogisticRegression_OHE,Добавление one-hot кодирования категориальных ...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.107333,0.070382,Логистическая регрессия на полном OHE-наборе п...
3,LogisticRegression_OHE,Проверка обобщающей способности OHE-модели на ...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.108667,0.072111,Test-качество для модели с OHE.
4,KNN_OHE_big_grid,"Расширенная настройка гиперпараметров KNN (k, ...",Pipeline(StandardScaler(with_mean=False) + KNe...,validation,0.116000,0.115699,Лучший набор гиперпараметров по weighted F1 на...
5,KNN_OHE_big_grid,Проверка обобщающей способности лучшей конфигу...,"n_neighbors=15, weights='distance', metric='mi...",test,0.106000,0.105685,"Качество на test ниже, чем на validation, что ..."
6,RandomForest_OHE_big_grid,Расширенная настройка гиперпараметров случайно...,"RandomForestClassifier(n_estimators=200, max_d...",validation,0.122000,0.121684,Лучший набор гиперпараметров по weighted F1 на...
7,RandomForest_OHE_big_grid,Проверка обобщающей способности лучшей конфигу...,"n_estimators=200, max_depth=None, min_samples_...",test,0.106667,0.106278,"На тесте качество ниже, чем на validation, воз..."


In [46]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

gb_params = []

for n_estimators in [50, 100, 200]:
    for learning_rate in [0.01, 0.05, 0.1, 0.2]:
        for max_depth in [2, 3, 5]:
            gb_params.append(
                {
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                    "max_depth": max_depth,
                }
            )

gb_results = []

for p in gb_params:
    gb = GradientBoostingClassifier(
        n_estimators=p["n_estimators"],
        learning_rate=p["learning_rate"],
        max_depth=p["max_depth"],
        random_state=42,
    )

    gb.fit(X_train_ohe, y_train)

    valid_pred = gb.predict(X_valid_ohe)
    test_pred  = gb.predict(X_test_ohe)

    valid_acc = accuracy_score(y_valid, valid_pred)
    test_acc  = accuracy_score(y_test,  test_pred)

    valid_f1 = f1_score(y_valid, valid_pred, average="weighted")
    test_f1  = f1_score(y_test,  test_pred,  average="weighted")

    gb_results.append(
        {
            **p,
            "valid_acc": valid_acc,
            "valid_f1": valid_f1,
            "test_acc": test_acc,
            "test_f1": test_f1,
        }
    )

gb_results_df = pd.DataFrame(gb_results)
gb_results_df.sort_values("valid_f1", ascending=False).head(10)

,n_estimators,learning_rate,max_depth,valid_acc,valid_f1,test_acc,test_f1
8,50,0.10,5,0.110667,0.108258,0.109333,0.107181
26,200,0.01,5,0.108667,0.107559,0.104667,0.100404
21,100,0.20,2,0.107333,0.106322,0.115333,0.112818
22,100,0.20,3,0.106000,0.106104,0.109333,0.108158
33,200,0.20,2,0.106667,0.105927,0.112667,0.111676
14,100,0.01,5,0.109333,0.104838,0.106667,0.097686
5,50,0.05,5,0.106000,0.104039,0.112000,0.108803
32,200,0.10,5,0.104000,0.103093,0.106000,0.105610
34,200,0.20,3,0.103333,0.102734,0.106667,0.106073
20,100,0.10,5,0.104000,0.102443,0.109333,0.108372


In [47]:
experiments.append(
    {
        "model_name": "GradientBoosting_OHE_big_grid",
        "hypothesis": "Настройка гиперпараметров градиентного бустинга по деревьям (число деревьев, глубина, шаг обучения) позволит улучшить качество по сравнению с логистической регрессией, KNN и случайным лесом.",
        "params": (
            "GradientBoostingClassifier(n_estimators=50, learning_rate=0.10, max_depth=5, random_state=42); "
            "n_estimators ∈ {50,100,200}, learning_rate ∈ {0.01,0.05,0.1,0.2}, max_depth ∈ {2,3,5}"
        ),
        "split": "validation",
        "accuracy": 0.110667,
        "f1_weighted": 0.108258,
        "comment": "Лучшая конфигурация градиентного бустинга по weighted F1 на validation; качество сопоставимо с KNN и случайным лесом.",
    }
)

In [48]:
experiments.append(
    {
        "model_name": "GradientBoosting_OHE_big_grid",
        "hypothesis": "Проверка обобщающей способности лучшей конфигурации градиентного бустинга на тестовом наборе.",
        "params": "n_estimators=50, learning_rate=0.10, max_depth=5, random_state=42",
        "split": "test",
        "accuracy": 0.109333,
        "f1_weighted": 0.107181,
        "comment": "Качество на test очень близко к validation, бустинг не даёт существенного выигрыша над другими моделями.",
    }
)

In [50]:
experiments_df = pd.DataFrame(experiments)
experiments_df

,model_name,hypothesis,params,split,accuracy,f1_weighted,comment
0,baseline,Простая логистическая регрессия без feature en...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.102667,0.074196,Нулевой baseline: качество чуть выше случайног...
1,baseline,Проверить обобщающую способность baseline на t...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.113333,0.081760,"Качество на test аналогично validation, baseli..."
2,LogisticRegression_OHE,Добавление one-hot кодирования категориальных ...,"LogisticRegression (solver=lbfgs, max_iter=100...",validation,0.107333,0.070382,Логистическая регрессия на полном OHE-наборе п...
3,LogisticRegression_OHE,Проверка обобщающей способности OHE-модели на ...,"LogisticRegression (solver=lbfgs, max_iter=100...",test,0.108667,0.072111,Test-качество для модели с OHE.
4,KNN_OHE_big_grid,"Расширенная настройка гиперпараметров KNN (k, ...",Pipeline(StandardScaler(with_mean=False) + KNe...,validation,0.116000,0.115699,Лучший набор гиперпараметров по weighted F1 на...
5,KNN_OHE_big_grid,Проверка обобщающей способности лучшей конфигу...,"n_neighbors=15, weights='distance', metric='mi...",test,0.106000,0.105685,"Качество на test ниже, чем на validation, что ..."
6,RandomForest_OHE_big_grid,Расширенная настройка гиперпараметров случайно...,"RandomForestClassifier(n_estimators=200, max_d...",validation,0.122000,0.121684,Лучший набор гиперпараметров по weighted F1 на...
7,RandomForest_OHE_big_grid,Проверка обобщающей способности лучшей конфигу...,"n_estimators=200, max_depth=None, min_samples_...",test,0.106667,0.106278,"На тесте качество ниже, чем на validation, воз..."
8,GradientBoosting_OHE_big_grid,Настройка гиперпараметров градиентного бустинг...,"GradientBoostingClassifier(n_estimators=50, le...",validation,0.110667,0.108258,Лучшая конфигурация градиентного бустинга по w...
9,GradientBoosting_OHE_big_grid,Проверка обобщающей способности лучшей конфигу...,"n_estimators=50, learning_rate=0.10, max_depth...",test,0.109333,0.107181,"Качество на test очень близко к validation, бу..."


In [ ]:
# склеиваем train и val
if hasattr(X_train_ohe, "toarray") and hasattr(X_valid_ohe, "toarray"):
    X_tv = np.vstack([X_train_ohe.toarray(), X_valid_ohe.toarray()])
else:
    X_tv = np.vstack([X_train_ohe, X_valid_ohe])

y_tv = np.concatenate([y_train, y_valid])

# лучшая модель по валидации
rf_best = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

# обучаем на train+val
rf_best.fit(X_tv, y_tv)

# считаем метрики только на тесте
test_pred = rf_best.predict(X_test_ohe)
test_acc  = accuracy_score(y_test, test_pred)
test_f1   = f1_score(y_test, test_pred, average="weighted")

test_acc, test_f1

k:\HSE\hseml-group-project-pe3ricia\myenv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


(0.10066666666666667, 0.0994082462580973)